# Post process and display results for LLM models


In [8]:
import json
import re
import copy as cp
import datetime as datetime
import pandas as pd
from functions_llm import *

In [9]:
#load results
MODEL_NAME =  "gpt-4o-mini"
savename = 'test_predict_llm'+MODEL_NAME+'.json'
file_path = '../Data_backup/results_llm/'+savename
# Open and read the JSON file
with open(file_path, 'r') as json_file:
    test_results_json = json.load(json_file)

In [10]:
#post process the results
def clean_output(results_json):
    dict_reports={}
    for report_id, report in results_json.items():
        results = report['results']
        results_process = cp.deepcopy(results)
        results_process = re.sub('[\{\}]', '', results_process)
        results_process= results_process.split("\n")
        results_process = results_process[1: -1]
        dict_results = dict()
        for pair in results_process:
            if len(pair.split(":")) == 2:
                key, value = pair.split(":")
            else:
                print(pair.split(":"))
                break
            key = re.sub("[\", ]","",key)
            value = re.sub("[\"]","",value)
            if key=='Location':
                #value = value
                #uncomment for parsing locations to list
                value = list(value.split(','))
                value = [re.sub("[\,, ]","",ival) for ival in value]
            elif key=='Start_Date' or key=='End_Date':
                value = value.replace(",","")
            elif key=='Sentences':
                pass
            else:
                value = re.sub("[\,, ]","",value)
            dict_results[key] = value
        dict_reports[report_id] = dict_results
    return dict_reports

In [23]:
dict_processed = clean_output(test_results_json)

['            "Pakistan has experienced an unusually intense and prolonged monsoon season, resulting in widespread infrastructure damage, numerous casualties, and significant injuries.",']
['    "Cameroons Far North region has been experiencing flooding since the start of the rainy season, which began in the second half of July.",']
['    "Intense rainfall observed in the departments of Mono, Couffo, Zou and Oum in the South of Benin caused the overflow of the river Couffo on 26 June 2024 in 6 of the 11 districts of the commune it crosses in Couffo department.",']


In [24]:
#parse to a panda dataframe
result_df_list = []
for report_id, report in dict_processed.items():
    results_df = pd.DataFrame(report)
    results_df['appealCode'] = report_id
    results_df.drop(['Sentences'], axis=1, inplace=True)
    result_df_list.append(results_df)
result_df_all = pd.concat(result_df_list)
result_df_all['Country'] = result_df_all['Country'].apply(lambda x: country_name_to_iso3(x))


In [62]:
result_df_all

,Hazard,Country,Location,Start_Date,End_Date,appealCode
0,Flood,DZA,"['Bchar', 'Elbayadh', 'BeniAbbes', 'Tamanrasse...",September 5 2024,NULL,MDRDZ011
1,Flood,DZA,"['Bchar', 'Elbayadh', 'BeniAbbes', 'Tamanrasse...",September 5 2024,NULL,MDRDZ011
2,Flood,DZA,"['Bchar', 'Elbayadh', 'BeniAbbes', 'Tamanrasse...",September 5 2024,NULL,MDRDZ011
3,Flood,DZA,"['Bchar', 'Elbayadh', 'BeniAbbes', 'Tamanrasse...",September 5 2024,NULL,MDRDZ011
4,Flood,DZA,"['Bchar', 'Elbayadh', 'BeniAbbes', 'Tamanrasse...",September 5 2024,NULL,MDRDZ011
5,Flood,DZA,"['Bchar', 'Elbayadh', 'BeniAbbes', 'Tamanrasse...",September 5 2024,NULL,MDRDZ011
0,Flood,PAK,"['Jacobabad', 'NaushahroFeroz', 'Ghotki', 'Suk...",July 2024,1 September 2024,MDRPK026
1,Flood,PAK,"['Jacobabad', 'NaushahroFeroz', 'Ghotki', 'Suk...",July 2024,1 September 2024,MDRPK026
2,Flood,PAK,"['Jacobabad', 'NaushahroFeroz', 'Ghotki', 'Suk...",July 2024,1 September 2024,MDRPK026
3,Flood,PAK,"['Jacobabad', 'NaushahroFeroz', 'Ghotki', 'Suk...",July 2024,1 September 2024,MDRPK026


In [26]:
#load labelled df for test
labelled_df = pd.read_csv('../Data_backup/results_llm/labelled_example.csv')
labelled_df.rename(columns={'Locations': 'Location'}, inplace=True)

In [27]:
labelled_df

,Hazard,Country,Location,Start_Date,End_Date,Name,appealCode
0,Flood,DZA,"['southern and western Algeria', 'Bchar, Elbay...","September 5, 2024","September 8, 2024",NaN,MDRDZ011
1,Flood,PAK,"['Balochistan ', 'Sindh ', 'Punjab', 'Khyber P...",July 2024,NaN,NaN,MDRPK026
2,Mass movement,PAK,"['KP, Azad Jammu and Kashmir AJK, and GilgitBa...",NaN,NaN,NaN,MDRPK026
3,Heat Wave,PAK,NaN,26 August 2024,1 September 2024,NaN,MDRPK026
4,Flood,CMR,"['Cameroons Far North region', 'Logone et Char...",second half of July 2024,"August 28, 2024",NaN,MDRCM039
5,Drought,CMR,NaN,2024,NaN,NaN,MDRCM039
6,Flood,BEN,"['Mono, Couffo, Zou and Oum in the South of Be...",26 June 2024,NaN,NaN,MDRBJ019
7,Flood,SDN,"['Red Sea, River Nile, and Northern State']",1 June 2024,12 August 2024,NaN,MDRSD034
8,Flood,NGA,"['Kano', 'Maiduguri', 'Bauchi state', 'Bauchi,...",8 August 2024,13 August 2024,NaN,MDRNG041
9,Flood,NGA,"['Sokoto State', 'Dantudu, Balakozo, Gidan Tud...",17 July 2024,NaN,NaN,MDRNG041


In [28]:
## calculate accuracy
precision_columns_list = ['Hazard', 'Country', 'Location', 'Start_Date', 'End_Date']
accuracy_dict = calculate_precision(result_df_all, labelled_df, precision_columns_list, unique_dict=unique_dict)

0
0
0
0
0


In [30]:
accuracy_dict

,Hazard,Country,Location,Start_Date,End_Date
0,NaN,NaN,NaN,NaN,NaN
